In [7]:
from fixture.factory.dataset.ohlcv import factory_ohlcv_cycle

df = factory_ohlcv_cycle()
df

Date,Open,High,Low,Close,Volume
datetime[μs],f64,f64,f64,f64,i64
2000-01-01 00:00:00,101.0,101.26,99.49,99.72,19103
2000-01-02 00:00:00,99.72,100.88,99.5,100.7,20967
2000-01-03 00:00:00,100.7,101.05,100.52,100.86,37362
2000-01-04 00:00:00,99.85,101.65,99.59,101.4,12924
2000-01-05 00:00:00,101.4,101.67,101.13,101.41,27734
…,…,…,…,…,…
2000-04-05 00:00:00,108.48,108.7,107.05,107.41,22066
2000-04-06 00:00:00,107.41,109.13,107.13,108.92,44474
2000-04-07 00:00:00,108.92,109.22,108.48,108.65,35380


In [8]:
from feature.closes.service import derive_closes


closes = derive_closes(df, 8)
closes

Date,now,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8
datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64
2000-01-10 00:00:00,-53.134042,51.161072,34.583306,10.893787,-48.435819,0.986145,53.396745,15.876169,97.795411
2000-01-11 00:00:00,13.802625,-53.134042,51.161072,34.583306,10.893787,-48.435819,0.986145,53.396745,15.876169
2000-01-12 00:00:00,72.641921,13.802625,-53.134042,51.161072,34.583306,10.893787,-48.435819,0.986145,53.396745
2000-01-13 00:00:00,22.470822,72.641921,13.802625,-53.134042,51.161072,34.583306,10.893787,-48.435819,0.986145
2000-01-14 00:00:00,-56.762728,22.470822,72.641921,13.802625,-53.134042,51.161072,34.583306,10.893787,-48.435819
…,…,…,…,…,…,…,…,…,…
2000-04-05 00:00:00,0.0,57.889984,-17.776119,42.152655,100.000833,64.682048,-13.351137,115.985005,-19.264117
2000-04-06 00:00:00,139.603802,0.0,57.889984,-17.776119,42.152655,100.000833,64.682048,-13.351137,115.985005
2000-04-07 00:00:00,-24.819611,139.603802,0.0,57.889984,-17.776119,42.152655,100.000833,64.682048,-13.351137


In [9]:
from feature.closes.derive import derive_closes_n4

closes_n4 = derive_closes_n4(df)
closes_n4

Date,now,lag_1,lag_2,lag_3,lag_4
datetime[μs],f64,f64,f64,f64,f64
2000-01-06 00:00:00,-48.435819,0.986145,53.396745,15.876169,97.795411
2000-01-07 00:00:00,10.893787,-48.435819,0.986145,53.396745,15.876169
2000-01-08 00:00:00,34.583306,10.893787,-48.435819,0.986145,53.396745
2000-01-09 00:00:00,51.161072,34.583306,10.893787,-48.435819,0.986145
2000-01-10 00:00:00,-53.134042,51.161072,34.583306,10.893787,-48.435819
…,…,…,…,…,…
2000-04-05 00:00:00,0.0,57.889984,-17.776119,42.152655,100.000833
2000-04-06 00:00:00,139.603802,0.0,57.889984,-17.776119,42.152655
2000-04-07 00:00:00,-24.819611,139.603802,0.0,57.889984,-17.776119


In [10]:
closes.equals(closes_n4)

False

In [11]:
# closesの特徴量分析

import plotly.express as px
import plotly.graph_objects as go
import polars as pl

# 相関行列ヒートマップ
corr = closes.drop("Date").to_pandas().corr()
fig = px.imshow(
    corr,
    labels=dict(x="Features", y="Features", color="Correlation"),
    x=corr.columns,
    y=corr.columns,
    title="特徴量間の相関関係ヒートマップ",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1
)
fig.update_layout(width=800, height=800)
fig.show()

# 特徴量分布のボックスプロット
fig = go.Figure()
for col in closes.columns[1:]:  # Dateを除外
    fig.add_trace(go.Box(
        y=closes[col],
        name=col,
        boxpoints='outliers',
        jitter=0.3,
        pointpos=-1.8
    ))
fig.update_layout(
    title="特徴量の分布と外れ値",
    yaxis_title="Value",
    boxmode='group'
)
fig.show()

# 時系列プロット（now特徴量）
fig = px.line(
    closes.to_pandas(),
    x="Date",
    y="now",
    title="'now'特徴量の時系列変化",
    labels={"now": "Value"},
    template="plotly_white"
)
# 移動平均を計算してプロット
fig.add_trace(go.Scatter(
    x=closes["Date"],
    y=closes["now"].rolling_mean(5),
    mode="lines",
    name="5日移動平均",
    line=dict(color="red", dash="dot")
))
fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Value",
    hovermode="x unified"
)
fig.show()

# ラグ特徴量の相互作用（3D散布図）
fig = px.scatter_3d(
    closes.head(100).to_pandas(),
    x='lag_1',
    y='lag_2',
    z='now',
    color='lag_3',
    title="ラグ特徴量の3D相互作用",
    labels={'lag_1': 'Lag 1', 'lag_2': 'Lag 2', 'now': 'Current'},
    color_continuous_scale=px.colors.sequential.Viridis
)
fig.update_layout(
    scene=dict(
        xaxis_title='Lag 1',
        yaxis_title='Lag 2',
        zaxis_title='Current Value'
    ),
    width=1000,
    height=800
)
fig.show()

In [ ]:
# 基本統計量の確認（修正版）
stats = closes.select([
    pl.all().exclude("Date").mean().name.suffix("_mean"),
    pl.all().exclude("Date").std().name.suffix("_std"),
    pl.all().exclude("Date").skew().name.suffix("_skew"),
    pl.all().exclude("Date").kurtosis().name.suffix("_kurt")
]).transpose(include_header=True, header_name="feature")
print("基本統計量:")
print(stats)

autocorr = closes.select([
    pl.corr(pl.col("now"), pl.col("now").shift(1)).alias("lag1"),
    pl.corr(pl.col("now"), pl.col("now").shift(2)).alias("lag2"),
    pl.corr(pl.col("now"), pl.col("now").shift(3)).alias("lag3"),
    pl.corr(pl.col("now"), pl.col("now").shift(4)).alias("lag4")
])
print("\n自己相関係数:")
print(autocorr)

# 分散比（前日との変動幅比較）
variance_ratio = closes.select([
    (pl.col("now").var() / pl.col("lag_1").var()).alias("var_ratio_now_vs_lag1"),
    (pl.col("lag_1").var() / pl.col("lag_2").var()).alias("var_ratio_lag1_vs_lag2")
])
print("\n分散比:")
print(variance_ratio)

# 異常値カウント（3σ以上）
outliers = closes.select([
    pl.col("now").filter(pl.col("now").abs() > 3 * pl.col("now").std()).count().alias("now_outliers"),
    pl.col("lag_1").filter(pl.col("lag_1").abs() > 3 * pl.col("lag_1").std()).count().alias("lag1_outliers")
])
print("\n異常値数（3σ以上）:")
print(outliers)

# トレンド継続日数分析
# trend_days = (
#     closes
#     .with_columns(
#         (pl.col("now") > 0).cast(pl.UInt8).alias("up_flag")
#     )
#     .with_columns(
#         pl.cumsum(
#             (pl.col("up_flag") != pl.col("up_flag").shift(1)).cast(pl.UInt8)
#         ).alias("run_id")
#     )
#     .groupby("run_id", "up_flag")
#     .agg(
#         consecutive_days = pl.count(),
#     )
#     .filter(pl.col("up_flag") == 1)
#     .select("consecutive_days")
# )
# print("\n上昇継続日数分布:")
# print(trend_days)

基本統計量:
shape: (36, 2)
┌────────────┬──────────┐
│ feature    ┆ column_0 │
│ ---        ┆ ---      │
│ str        ┆ f64      │
╞════════════╪══════════╡
│ now_mean   ┆ 7.974909 │
│ lag_1_mean ┆ 8.044585 │
│ lag_2_mean ┆ 7.990572 │
│ lag_3_mean ┆ 8.383027 │
│ lag_4_mean ┆ 6.316657 │
│ …          ┆ …        │
│ lag_4_kurt ┆ 0.409794 │
│ lag_5_kurt ┆ 0.410017 │
│ lag_6_kurt ┆ 0.414287 │
│ lag_7_kurt ┆ 0.42323  │
│ lag_8_kurt ┆ 0.364373 │
└────────────┴──────────┘

自己相関係数:
shape: (1, 4)
┌───────────┬──────────┬──────────┬──────────┐
│ lag1      ┆ lag2     ┆ lag3     ┆ lag4     │
│ ---       ┆ ---      ┆ ---      ┆ ---      │
│ f64       ┆ f64      ┆ f64      ┆ f64      │
╞═══════════╪══════════╪══════════╪══════════╡
│ -0.333685 ┆ 0.217324 ┆ 0.157276 ┆ 0.143055 │
└───────────┴──────────┴──────────┴──────────┘

分散比:
shape: (1, 2)
┌───────────────────────┬────────────────────────┐
│ var_ratio_now_vs_lag1 ┆ var_ratio_lag1_vs_lag2 │
│ ---                   ┆ ---                    │
│ f64      

AttributeError: module 'polars' has no attribute 'cumsum'